In [ ]:
import joblib
import pandas as pd

In [ ]:
# model laden
model = joblib.load("kijkcijfer_model.pkl")

In [ ]:
# RATINGS DATA

ratings_df = pd.read_csv('input.csv')

# Verander de notatie van getallen (punten gebruiken als decimaalteken in plaats van komma's)
ratings_df['Kijkers'] = ratings_df['Kijkers'].str.replace('.', '').str.replace(',', '.')

# Eerst omzetten naar float (om decimalen te kunnen verwerken). Niet-numerieke waarden worden een NaN
ratings_df['Kijkers'] = pd.to_numeric(ratings_df['Kijkers'], errors='coerce')

# Verwijder NaN waarden
print(f"Number of rows with non-numeric values: {ratings_df['Kijkers'].isna().sum()}")
ratings_df = ratings_df.dropna(subset=['Kijkers'])

# Omzetten naar integer (dit rondt de decimalen af)
ratings_df['Kijkers'] = ratings_df['Kijkers'].astype(int)

# Datum omzetten in datetime
ratings_df["Datum"] = pd.to_datetime(ratings_df["Datum"])

# Start omzetten in datetime
ratings_df["Start"] = pd.to_datetime(ratings_df["Start"]).dt.time
ratings_df["Start"] = [pd.Timestamp.combine(d,t) for d,t in zip(ratings_df['Datum'],ratings_df['Start'])]

# Duur omzetten in timedelta
ratings_df["Duur"] = pd.to_timedelta(ratings_df["Duur"])

# Zet verschillende waardes voor EEN om
ratings_df.loc[ratings_df["Zender"].isin(["EEN", "VRT 1"]), "Zender"] = "EEN"

# Zet verschillende waardes voor CANVAS om
ratings_df.loc[ratings_df["Zender"].isin(["Canvas", "CANVAS", "VRT CANVAS"]), "Zender"] = "CANVAS"

# Zet verschillende waardes voor KETNET om
ratings_df.loc[ratings_df["Zender"].isin(["KETNET", "OP 12"]), "Zender"] = "KETNET"

# Zet verschillende waardes voor PLAY4 om
ratings_df.loc[ratings_df["Zender"].isin(["VIER", "PLAY4"]), "Zender"] = "PLAY4"

# Zet verschillende waardes voor PLAY5 om
ratings_df.loc[ratings_df["Zender"].isin(["VIJF", "PLAY5"]), "Zender"] = "PLAY5"

# Zet verschillende waardes voor PLAY6 om
ratings_df.loc[ratings_df["Zender"].isin(["ZES", "PLAY6"]), "Zender"] = "PLAY6"

# Zet verschillende waardes voor VTM2 om
ratings_df.loc[ratings_df["Zender"].isin(["Q2", "VTM2"]), "Zender"] = "VTM2"

# Zet verschillende waardes voor VTM3 om
ratings_df.loc[ratings_df["Zender"].isin(["VITAYA", "VTM3"]), "Zender"] = "VTM3"

# Zet verschillende waardes voor VTM4 om
ratings_df.loc[ratings_df["Zender"].isin(["CAZ", "VTM4"]), "Zender"] = "VTM4"

# Wijs programma's over gedeelde zenders toe aan EEN
ratings_df.loc[ratings_df["Zender"].isin(["EEN,VTM,PLAY4", "EEN, VTM, PLAY", "VRT 1/VTM/Play4"]), "Zender"] = "EEN"

# Geef ELEVEN en DAZN dezelfde waarde
ratings_df.loc[ratings_df["Zender"].isin(["ELEVEN PRO LEAGUE 1 NL", "DAZN PRO LEAGUE 1 (NL)"]), "Zender"] = "PRO LEAGUE 1"

In [ ]:
# ZON DATA
solar_df = pd.DataFrame({
    'Datum': [
        '2025-06-01',
        '2025-06-02',
        '2025-06-03',
        '2025-06-04',
        '2025-06-05',
        '2025-06-06',
        '2025-06-07',
        '2025-06-08',
        '2025-06-09',
        '2025-06-10',
        '2025-06-11',
        '2025-06-12',
        '2025-06-13',
        '2025-06-14',
    ],
    'Zonsopgang': [
        '2025-06-01 05:34:37'
        '2025-06-02 05:33:53'
        '2025-06-03 05:33:12'
        '2025-06-04 05:32:34'
        '2025-06-05 05:31:58'
        '2025-06-06 05:31:26'
        '2025-06-07 05:30:56'
        '2025-06-08 05:30:29'
        '2025-06-09 05:30:05'
        '2025-06-10 05:29:45'
        '2025-06-11 05:29:27'
        '2025-06-12 05:29:12'
        '2025-06-13 05:29:00'
        '2025-06-14 05:28:51'
    ],
    'Zonsondergang': [
        '2025-06-01 21:46:58'
        '2025-06-02 21:47:59'
        '2025-06-03 21:48:58'
        '2025-06-04 21:49:55'
        '2025-06-05 21:50:50'
        '2025-06-06 21:51:43'
        '2025-06-07 21:52:33'
        '2025-06-08 21:53:21'
        '2025-06-09 21:54:07'
        '2025-06-10 21:54:50'
        '2025-06-11 21:55:30'
        '2025-06-12 21:56:08'
        '2025-06-13 21:56:43'
        '2025-06-14 21:57:15'
    ]
})

# Datums omzetten
solar_df["Datum"] = pd.to_datetime(solar_df["Datum"])
solar_df["Zonsopgang"] = pd.to_datetime(solar_df["Zonsopgang"])
solar_df["Zonsondergang"] = pd.to_datetime(solar_df["Zonsondergang"])

# Mergen met ratings data
ratings_solar_df = pd.merge(ratings_df, solar_df, on="Datum", how="inner")

In [ ]:
# GEMIDDELDE WEER DATA BEREKENEN

weather_df = pd.read_csv('data/aws_1day.csv')

# Hou nuttige kolommen over
weather_df = weather_df[['timestamp', 'temp_avg', 'sun_duration', 'precip_quantity', 'pressure', 'humidity_rel_shelter_avg']]
weather_df.rename(columns={
    "timestamp": "Datum",
    "temp_avg": "Temp",
    "sun_duration": "Zon",
    "precip_quantity": "Regen",
    "pressure": "Druk",
    "humidity_rel_shelter_avg": "Luchtvochtigheid"
}, inplace=True)

# Zet datum om in het juiste format
weather_df["Datum"] = pd.to_datetime(weather_df["Datum"])
weather_df["Maand"] = weather_df["Datum"].dt.month
weather_df["Dag"] = weather_df["Datum"].dt.day

# Filter data voor de maand juni
weather_df = weather_df[weather_df["Maand"] == 6]

# Groepeer data per dag en neem gemiddeldes van elke kolom
weather_df = weather_df.groupby("Dag").mean().reset_index()

weather_df

In [ ]:
# WEER DATA
weather_df = pd.DataFrame({
    'Datum': [
        '2025-06-01',
        '2025-06-02',
        '2025-06-03',
        '2025-06-04',
        '2025-06-05',
        '2025-06-06',
        '2025-06-07',
        '2025-06-08',
        '2025-06-09',
        '2025-06-10',
        '2025-06-11',
        '2025-06-12',
        '2025-06-13',
        '2025-06-14',
    ],
    'Temp': [''],
    'Zon': [''],
    'Regen': [''],
    'Druk': [''],
    'Luchtvochtigheid': [''],
})

# Zet datum om in het juiste format
weather_df["Datum"] = pd.to_datetime(weather_df["Datum"])

# Merge met ratings_solar_df
data = pd.merge(ratings_solar_df, weather_df, on="Datum", how="inner")

In [ ]:
# FEATURE ENGINEERING

# Jaar, maand, dag en week afleiden uit de datum
data["Jaar"] = data["Datum"].dt.year
data["Maand"] = data["Datum"].dt.month
data["Dag"] = data["Datum"].dt.day_of_week
data["Week"] = data["Datum"].dt.isocalendar().week

# Covid features
data["Covid19"] = np.where((data["Datum"] >= "2020-02-04") & (data["Datum"] <= "2022-03-13"), 1, 0) # Van de eerste nationale veiligheidsraad tot wanneer we naar code geel gingen
data["Lockdown1"] = np.where((data["Datum"] >= "2020-03-13") & (data["Datum"] <= "2020-06-08"), 1, 0) # Afbouwplan startte op 8 juni.
data["Lockdown2"] = np.where((data["Datum"] >= "2020-10-30") & (data["Datum"] <= "2021-04-19"), 1, 0) # Minder duidelijk afbouwplan.Het onderwijs herstart terug op 19 april.

# Eind tijd berekenen
data["Eind"] = data["Start"] + data["Duur"]

# Startuur en einduur afsplitsen
data["Startuur"] = data["Start"].dt.hour
data["Einduur"] = data["Eind"].dt.hour

# Duur in minuten berekenen
data["Duur"] = np.round(data["Duur"].dt.total_seconds() / 60)
data["Duur"] = data["Duur"].astype(int)

# Voeg primetime kolommen toe
data["Primetime"] = ((data["Startuur"] >= 20) & (data["Einduur"] <= 22)).astype(int)
data["Einde_In_Primetime"] = ((data["Einduur"] >= 20) & (data["Einduur"] <= 22)).astype(int)
data["Start_In_Primetime"] = ((data["Startuur"] >= 20) & (data["Startuur"] <= 22)).astype(int)

# Zenders van de openbare omroep onderbreken hun programma's niet voor reclame
data["Reclame_Onderbrekingen"] = np.where(data["Zender"].isin(["EEN", "CANVAS", "KETNET", "LA UNE"]), 0, 1)

# Delta's zonsopgang/zonsondergang
data["Zonsopgang_Delta_Start"] = data["Startuur"] - data["Zonsopgang"].dt.hour
data["Zonsopgang_Delta_Einde"] = data["Einduur"] - data["Zonsopgang"].dt.hour
data["Zonsondergang_Delta_Start"] = data["Zonsondergang"].dt.hour - data["Startuur"]
data["Zonsondergang_Delta_Einde"] = data["Zonsondergang"].dt.hour - data["Einduur"]

# Sommige features hebben we niet nodig, we kunnen de andere ook wel beter sorteren
final_features = ["Programma", "Zender", "Jaar", "Maand", "Dag", "Week", "Covid19", "Lockdown1", "Lockdown2", "Startuur", "Einduur", "Duur", "Primetime", "Einde_In_Primetime", "Start_In_Primetime", "Zonsopgang_Delta_Start", "Zonsopgang_Delta_Einde", "Zonsondergang_Delta_Start", "Zonsondergang_Delta_Einde", "Reclame_Onderbrekingen", "Temp", "Zon", "Regen", "Druk", "Luchtvochtigheid", "Kijkers"]
data = data[final_features]

# Drop duplicates
data = data.drop_duplicates()

data.info()

In [ ]:
# VOORSPELLINGEN MAKEN

new_data = data.drop(columns=["Programma"])

predictions = model.predict(new_data)
predictions

In [ ]:
# Resultaten toevoegen aan data df
data["Kijkers"] = predictions
data

In [ ]:
# Opslaan als csv
csv = data.to_csv("examen_oplossing.csv", index=False)